#  Objective -    This code is built for preparation of building statistical models using the backtesting process

#  Description -  Code uses Random Forrest, SVM & CNN for bulding models

# Date  -   15th July 2021

# Author -       Avinash Mishra

# ----Importing the libraries ----#

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import clear_output
from numpy import arange
import datetime as dt

import fnmatch
import os

from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report, confusion_matrix

import time


In [ ]:
import import_ipynb

import RandomForest_with_pickle_dump
import NeuralNetwork_with_pickle_dump 
import SVM_with_pickle_dump 


# -------Importing the Iteration List -----#

In [ ]:
method_list = pd.read_excel("Iteration List 2.xlsx")
method_list = method_list[(method_list['Technique'] == 'RandomForest')&(method_list['Strategy'] == 'ClientEma')]
method_list.reset_index(inplace=True)
method_list.drop('index',axis=1,inplace=True)
method_list = method_list.loc[:9]
method_list

In [ ]:
#-----Data File Path Name ----#
data_path_name = "<local-path>"


#------Classifier data path ------#
classifier_path_name = "<local-path>"


#------Relative importance data path ------#
rel_imp_path_name = "<local-path>"



#-------Data Path for Creation of Folders & Subfolders --------#
folder_path_name = "<local-path>"

# ----Model Probability Vs (1/RR_Ratio+1) ----

In [ ]:
def model_probab_vs_RR_Ratio(final_dataset_4):



    year = ['Overall',2019,2020]

    list_a = ['RR_ratio_without_discount','RR_ratio_discount_5','RR_ratio_discount_10','RR_ratio_discount_15',
              'RR_ratio_discount_20','RR_ratio_plus_5','RR_ratio_plus_10','RR_ratio_plus_15','RR_ratio_plus_20']



    df_probab = pd.DataFrame()
    i = 0

    for i_year in year:

        if i_year == 'Overall':
            final_dataset_4_a = final_dataset_4
        elif i_year == 2019:
            final_dataset_4_a = final_dataset_4[final_dataset_4['Year'] == 2019]
            final_dataset_4_a['index'] = final_dataset_4_a.index
        elif i_year == 2020:
            final_dataset_4_a = final_dataset_4[final_dataset_4['Year'] == 2020]
            final_dataset_4_a['index'] = final_dataset_4_a.index



        df_2_test =final_dataset_4_a.groupby(by=['index','Condition','Currency','Timeframe'])['Actual_Good_Bad','Probability__Good','Profit','Risk_Reward_Ratio'].mean()
        df_2_test.reset_index(inplace = True)

        df_2_test['RR_ratio'] = 1/(df_2_test['Risk_Reward_Ratio']+1)


        df_2_test['RR_ratio_without_discount'] = df_2_test['RR_ratio']
        df_2_test['RR_ratio_discount_5'] = df_2_test['RR_ratio'] - (5/100)
        df_2_test['RR_ratio_discount_10'] = df_2_test['RR_ratio'] - (10/100)
        df_2_test['RR_ratio_plus_5'] = df_2_test['RR_ratio'] + (5/100)
        df_2_test['RR_ratio_plus_10'] = df_2_test['RR_ratio'] + (10/100)

        df_2_test['RR_ratio_plus_15'] = df_2_test['RR_ratio'] + (15/100)
        df_2_test['RR_ratio_plus_20'] = df_2_test['RR_ratio'] + (20/100)

        df_2_test['RR_ratio_discount_15'] = df_2_test['RR_ratio'] - (15/100)
        df_2_test['RR_ratio_discount_20'] = df_2_test['RR_ratio'] - (20/100)



        for item in list_a:
            print(item)
            df_probab.loc[i,'Year'] = i_year

            df_probab.loc[i,'RR_ratios'] = item

            df_probab.loc[i,'Model_Profit'] = df_2_test[df_2_test['Probability__Good'] >= df_2_test[item]]['Profit'].sum()
            df_probab.loc[i,'Base_Profit'] = df_2_test['Profit'].sum()

            df_probab.loc[i,'Model_Predicted_Good'] = df_2_test[df_2_test['Probability__Good'] >= df_2_test[item]]['Actual_Good_Bad'].sum()
            df_probab.loc[i,'Base_Total_Good'] = df_2_test['Actual_Good_Bad'].sum()

            df_probab.loc[i,'Model_Total_trades'] = len(df_2_test[df_2_test['Probability__Good'] >= df_2_test[item]])
            df_probab.loc[i,'Base_Total_trades'] = len(df_2_test)

            df_probab.loc[i,'Model_Accuracy'] = df_probab.loc[i,'Model_Predicted_Good']/df_probab.loc[i,'Model_Total_trades']
            df_probab.loc[i,'Base_Accuracy'] = df_probab.loc[i,'Base_Total_Good']/df_probab.loc[i,'Base_Total_trades']

            df_probab.loc[i,'Model Trades Pass Through'] = df_probab.loc[i,'Model_Total_trades']/df_probab.loc[i,'Base_Total_trades']
            df_probab.loc[i,'Percent Profit Covered'] = df_probab.loc[i,'Model_Profit']/df_probab.loc[i,'Base_Profit']
            
            df_probab.loc[i,'Lift'] = df_probab.loc[i,'Percent Profit Covered'] - df_probab.loc[i,'Model Trades Pass Through']
            
            


            i+=1
    
    
    return df_probab

# --Percentile Table--

In [ ]:
def percentile_table(dataset):
    
    data_percentile = dataset
    
    year = ['Overall',2019,2020]
    p_table = pd.DataFrame()
    i=0
    
    
    for i_year in year:
    
        if i_year == 'Overall':
            final_dataset_3 = data_percentile
        elif i_year == 2019:
            final_dataset_3 = data_percentile[data_percentile['Year'] == 2019]
            final_dataset_3['index'] = final_dataset_3.index
        elif i_year == 2020:
            final_dataset_3 = data_percentile[data_percentile['Year'] == 2020]
            final_dataset_3['index'] = final_dataset_3.index

    
    
        list_a = [10,20,30,40,50,60,70,80,90]
        for n in list_a:
            p_table.loc[i,'Year'] = i_year

            p_table.loc[i,'Percentile'] = n

            p_table.loc[i,'Cut_Off_Probability'] = np.percentile(np.array(final_dataset_3['Probability__Good']),n)

            cut_off = p_table.loc[i,'Cut_Off_Probability']

            p_table.loc[i,'TN'] = len(final_dataset_3[final_dataset_3['Actual_Good_Bad'] == 0]\
                                      [final_dataset_3['Probability__Good'] < cut_off])

            p_table.loc[i,'FP'] = len(final_dataset_3[final_dataset_3['Actual_Good_Bad'] == 0]\
                                      [final_dataset_3['Probability__Good'] >= cut_off])

            p_table.loc[i,'FN'] = len(final_dataset_3[final_dataset_3['Actual_Good_Bad'] == 1]\
                                      [final_dataset_3['Probability__Good'] < cut_off])

            p_table.loc[i,'TP'] = len(final_dataset_3[final_dataset_3['Actual_Good_Bad'] == 1]\
                                      [final_dataset_3['Probability__Good'] >= cut_off])

            p_table.loc[i,'Total'] = p_table.loc[i,'TN'] + p_table.loc[i,'FP'] + p_table.loc[i,'FN'] + p_table.loc[i,'TP']

            p_table.loc[i,'FP+TP'] = p_table.loc[i,'FP'] + p_table.loc[i,'TP']

            p_table.loc[i,'Trades Pass Through'] = p_table.loc[i,'FP+TP']/p_table.loc[i,'Total']

            p_table.loc[i,'Model Profit'] = (final_dataset_3[final_dataset_3['Probability__Good'] >= cut_off]['Profit']).sum()
            
            p_table.loc[i,'Base Profit'] = (final_dataset_3['Profit']).sum()
            
            p_table.loc[i,'Percent Profit Covered'] = p_table.loc[i,'Model Profit']/p_table.loc[i,'Base Profit']

            p_table.loc[i,'Predicted 1 Accuracy'] = p_table.loc[i,'TP']/(p_table.loc[i,'TP'] + p_table.loc[i,'FP'])

            p_table.loc[i,'Base_1s'] = len(final_dataset_3[final_dataset_3['Actual_Good_Bad'] == 1]\
                                           [final_dataset_3['Probability__Good'] >= cut_off])

            p_table.loc[i,'Base_0s'] = len(final_dataset_3[final_dataset_3['Actual_Good_Bad'] == 0]\
                                           [final_dataset_3['Probability__Good'] >= cut_off])
            
            p_table.loc[i,'Base Accuracy'] = (p_table.loc[i,'TP'] + p_table.loc[i,'FN'])/(p_table.loc[i,'TN'] + p_table.loc[i,'FP'] + p_table.loc[i,'FN'] + p_table.loc[i,'TP'])

            p_table.loc[i,'Base Trades Pass Through'] = p_table.loc[i,'Base_1s'] + p_table.loc[i,'Base_0s']
            
            p_table.loc[i,'Lift'] = p_table.loc[i,'Percent Profit Covered'] - p_table.loc[i,'Trades Pass Through']

            i+=1
        
    return p_table

# ---Train Rank 1 vs test---

In [ ]:
def cal_profit_2(data,year):
    group_df = pd.DataFrame()
    df1 = pd.DataFrame()
    df_result = pd.DataFrame()
    df1['Condition'] = data['Condition']
    df1['Currency'] = data['Currency']
    df1['Timeframe'] = data['Timeframe']
    df1['Profit'] = data['Profit']
    df1['actual'] = data['Actual_Good_Bad']
    df1['predicted_prob'] = data['Probability__Good']
    
 
    
    for n in arange(0.05,1,0.05):
        

        df1['Threshold_Value'] = n
        df1['Predicted_Good'] = np.where(df1['predicted_prob']>n,1,0)
        
        df1['False_Postive'] =np.where((df1['actual'] == 0)&(df1['Predicted_Good']==1),1,0)
        df1['True_Postive'] = np.where((df1['actual'] ==1) & (df1['Predicted_Good']==1),1,0)
            
        
        df1['Flag1_Profit'] = (df1['False_Postive'] + df1['True_Postive'])*df1['Profit']
        
        
        
        
        df_result = pd.concat([df_result,df1],axis = 0,ignore_index=True)
        
    
    group_df = pd.DataFrame()
    group_df['Sum_TP'] = df_result.groupby(['Currency','Condition','Timeframe','Threshold_Value'])["True_Postive"].sum()
    group_df['Sum_FP'] = df_result.groupby(['Currency','Condition','Timeframe','Threshold_Value'])["False_Postive"].sum()
    group_df['Model_Profit'] = df_result.groupby(['Currency','Condition','Timeframe','Threshold_Value'])["Flag1_Profit"].sum()
    group_df['Base_Profit'] = df_result.groupby(['Currency','Condition','Timeframe','Threshold_Value'])['Profit'].sum()
    
    
    
    
    group_df['Actual_Good'] = df_result.groupby(['Currency','Condition','Timeframe','Threshold_Value'])['actual'].sum()
    group_df['Actual_Total_Trades'] = df_result.groupby(['Currency','Condition','Timeframe','Threshold_Value'])['actual'].count()
    
    group_df.reset_index(inplace=True)
    
    group_df['Base_Accuracy'] = group_df['Actual_Good']/group_df['Actual_Total_Trades']
    
    group_df['Year'] = year
    
        
    return df_result,group_df


In [ ]:
def Ranks(final_dataset_4,final_dataset_4_train):


    year = ['Overall',2019,2020]
    x_test = pd.DataFrame()

    for i_year in year:

        if i_year == 'Overall':
            final_dataset_4_a = final_dataset_4
        elif i_year == 2019:
            final_dataset_4_a = final_dataset_4[final_dataset_4['Year'] == 2019]
            final_dataset_4_a['index'] = final_dataset_4_a.index
        elif i_year == 2020:
            final_dataset_4_a = final_dataset_4[final_dataset_4['Year'] == 2020]
            final_dataset_4_a['index'] = final_dataset_4_a.index


        df_1_test =final_dataset_4_a.groupby(by=['index','Condition','Currency','Timeframe'])['Actual_Good_Bad','Probability__Good','Profit'].mean()
        df_1_test.reset_index(inplace=True)
        x = cal_profit_2(df_1_test,i_year)




        x_test = pd.concat([x_test,x[1]],axis = 0,ignore_index=True)


    x_test["Rank_wrt_Profit"] = x_test.groupby(['Year','Currency','Condition','Timeframe'])["Model_Profit"].rank(method='first',ascending=False)


    df_1_test_train =final_dataset_4_train.groupby(by=['index','Condition','Currency','Timeframe'])['Actual_Good_Bad','Probability__Good','Profit'].mean()
    df_1_test_train.reset_index(inplace=True)
    x_train = cal_profit_2(df_1_test_train,"Train")
    x_train[1]["Rank_wrt_Profit"] = x_train[1].groupby(['Currency','Condition','Timeframe'])["Model_Profit"].rank(method='first',ascending=False)
    
    
    
    return x_test,x_train[1]

# -------Currency,Condition - Model Probab vs 1/Risk Ratio+1 Table----#

In [ ]:
def Curr_Cond_Model_Probab_vs_RR_Ratio(final_dataset_4):

    year = ['Overall',2019,2020]

    list_a = ['RR_ratio_without_discount','RR_ratio_discount_5','RR_ratio_discount_10','RR_ratio_discount_15',\
              'RR_ratio_discount_20','RR_ratio_plus_5','RR_ratio_plus_10','RR_ratio_plus_15','RR_ratio_plus_20']


    df_probab = pd.DataFrame()
    i = 0



    for i_year in year:
        #print(i_year)

        if i_year == 'Overall':
            final_dataset_4_a = final_dataset_4
        elif i_year == 2019:
            final_dataset_4_a = final_dataset_4[final_dataset_4['Year'] == 2019]
            final_dataset_4_a['index'] = final_dataset_4_a.index
        elif i_year == 2020:
            final_dataset_4_a = final_dataset_4[final_dataset_4['Year'] == 2020]
            final_dataset_4_a['index'] = final_dataset_4_a.index



        df_2_test =final_dataset_4_a.groupby(by=['index','Condition','Currency','Timeframe'])['Actual_Good_Bad','Probability__Good','Profit','Risk_Reward_Ratio'].mean()
        df_2_test.reset_index(inplace = True)

        df_2_test['RR_ratio'] = 1/(df_2_test['Risk_Reward_Ratio']+1)


        df_2_test['RR_ratio_without_discount'] = df_2_test['RR_ratio']
        df_2_test['RR_ratio_discount_5'] = df_2_test['RR_ratio'] - (5/100)
        df_2_test['RR_ratio_discount_10'] = df_2_test['RR_ratio'] - (10/100)
        df_2_test['RR_ratio_plus_5'] = df_2_test['RR_ratio'] + (5/100)
        df_2_test['RR_ratio_plus_10'] = df_2_test['RR_ratio'] + (10/100)
        df_2_test['RR_ratio_plus_15'] = df_2_test['RR_ratio'] + (15/100)
        df_2_test['RR_ratio_plus_20'] = df_2_test['RR_ratio'] + (20/100)
        df_2_test['RR_ratio_discount_15'] = df_2_test['RR_ratio'] - (15/100)
        df_2_test['RR_ratio_discount_20'] = df_2_test['RR_ratio'] - (20/100)



        for itr,r in final_iter_list.iterrows():
            curr = r['Currency']
            cond = r['Condition']
            t_frame = r['TimeFrame']


            for item in list_a:
                #print(item)
                df_probab.loc[i,'Year'] = i_year

                df_probab.loc[i,'RR_ratios'] = item

                df_probab.loc[i,'Currency'] = curr

                df_probab.loc[i,'Condition'] = cond

                df_probab.loc[i,'Timeframe'] = t_frame



                df_probab.loc[i,'Model_Profit'] = df_2_test[(df_2_test['Condition']==cond)&\
                                                            (df_2_test['Currency']==curr)&\
                                                            (df_2_test['Timeframe']==t_frame)&\
                                                            (df_2_test['Probability__Good'] >= df_2_test[item])]['Profit'].sum()


                df_probab.loc[i,'Base_Profit'] = df_2_test[(df_2_test['Condition']==cond)&\
                                                            (df_2_test['Currency']==curr)&\
                                                            (df_2_test['Timeframe']==t_frame)]['Profit'].sum()


                df_probab.loc[i,'Model_Predicted_Good'] = df_2_test[(df_2_test['Condition']==cond)&\
                                                                    (df_2_test['Currency']==curr)&\
                                                                    (df_2_test['Timeframe']==t_frame)&\
                                                                    (df_2_test['Probability__Good'] >= df_2_test[item])]['Actual_Good_Bad'].sum() 
                df_probab.loc[i,'Base_Total_Good'] = df_2_test[(df_2_test['Condition']==cond)&\
                                                            (df_2_test['Currency']==curr)&\
                                                            (df_2_test['Timeframe']==t_frame)]['Actual_Good_Bad'].sum()


                df_probab.loc[i,'Model_Total_trades'] = len(df_2_test[(df_2_test['Condition']==cond)&\
                                                                      (df_2_test['Currency']==curr)&\
                                                                      (df_2_test['Timeframe']==t_frame)&\
                                                                      (df_2_test['Probability__Good'] >= df_2_test[item])])
                df_probab.loc[i,'Base_Total_trades'] = len(df_2_test[(df_2_test['Condition']==cond)&\
                                                                     (df_2_test['Currency']==curr)&\
                                                                     (df_2_test['Timeframe']==t_frame)])

                df_probab.loc[i,'Model_Accuracy'] = df_probab.loc[i,'Model_Predicted_Good']/df_probab.loc[i,'Model_Total_trades']
                df_probab.loc[i,'Base_Accuracy'] = df_probab.loc[i,'Base_Total_Good']/df_probab.loc[i,'Base_Total_trades']

                df_probab.loc[i,'Model Trades Pass Through'] = df_probab.loc[i,'Model_Total_trades']/df_probab.loc[i,'Base_Total_trades']
                df_probab.loc[i,'Percent Profit Covered'] = df_probab.loc[i,'Model_Profit']/df_probab.loc[i,'Base_Profit']
                
                df_probab.loc[i,'Lift'] = df_probab.loc[i,'Percent Profit Covered'] - df_probab.loc[i,'Model Trades Pass Through']


                i+=1


    df_probab = df_probab.sort_values(by = ['Year','RR_ratios','Currency','Condition','Timeframe'])
    
    return df_probab

# -----Currency,Condition - Percentile Table---#

In [ ]:
def curr_cond_percentile_table(dataset):
    
    data_percentile = dataset
    
    year = ['Overall',2019,2020]
    p_table = pd.DataFrame()
    i=0
    
    
    for i_year in year:
        #print(i_year)
    
        if i_year == 'Overall':
            final_dataset_3 = data_percentile
        elif i_year == 2019:
            final_dataset_3 = data_percentile[data_percentile['Year'] == 2019]
            final_dataset_3['index'] = final_dataset_3.index
        elif i_year == 2020:
            final_dataset_3 = data_percentile[data_percentile['Year'] == 2020]
            final_dataset_3['index'] = final_dataset_3.index

    
    
        list_a = [10,20,30,40,50,60,70,80,90]
        
        
        for itr,r in final_iter_list.iterrows():
            curr = r['Currency']
            cond = r['Condition']
            t_frame = r['TimeFrame']
                
        
            for n in list_a:
                p_table.loc[i,'Year'] = i_year

                p_table.loc[i,'Percentile'] = n
                
                p_table.loc[i,'Currency'] = curr

                p_table.loc[i,'Condition'] = cond

                p_table.loc[i,'Timeframe'] = t_frame

                p_table.loc[i,'Cut_Off_Probability'] = np.percentile(np.array(final_dataset_3['Probability__Good']),n)

                cut_off = p_table.loc[i,'Cut_Off_Probability']

                p_table.loc[i,'TN'] = len(final_dataset_3[(final_dataset_3['Condition']==cond)&\
                                                          (final_dataset_3['Currency']==curr)&\
                                                          (final_dataset_3['Timeframe']==t_frame)&\
                                                          (final_dataset_3['Actual_Good_Bad'] == 0)]\
                                          [final_dataset_3['Probability__Good'] < cut_off])

                p_table.loc[i,'FP'] = len(final_dataset_3[(final_dataset_3['Condition']==cond)&\
                                                          (final_dataset_3['Currency']==curr)&\
                                                          (final_dataset_3['Timeframe']==t_frame)&\
                                                          (final_dataset_3['Actual_Good_Bad'] == 0)]\
                                          [final_dataset_3['Probability__Good'] >= cut_off])

                p_table.loc[i,'FN'] = len(final_dataset_3[(final_dataset_3['Condition']==cond)&\
                                                          (final_dataset_3['Currency']==curr)&\
                                                          (final_dataset_3['Timeframe']==t_frame)&\
                                                          (final_dataset_3['Actual_Good_Bad'] == 1)]\
                                          [final_dataset_3['Probability__Good'] < cut_off])

                p_table.loc[i,'TP'] = len(final_dataset_3[(final_dataset_3['Condition']==cond)&\
                                                          (final_dataset_3['Currency']==curr)&\
                                                          (final_dataset_3['Timeframe']==t_frame)&\
                                                          (final_dataset_3['Actual_Good_Bad'] == 1)]\
                                          [final_dataset_3['Probability__Good'] >= cut_off])

                p_table.loc[i,'Total'] = p_table.loc[i,'TN'] + p_table.loc[i,'FP'] + p_table.loc[i,'FN'] + p_table.loc[i,'TP']

                p_table.loc[i,'FP+TP'] = p_table.loc[i,'FP'] + p_table.loc[i,'TP']

                p_table.loc[i,'Trades Pass Through'] = p_table.loc[i,'FP+TP']/p_table.loc[i,'Total']

                p_table.loc[i,'Model Profit'] = (final_dataset_3[(final_dataset_3['Condition']==cond)&\
                                                                 (final_dataset_3['Currency']==curr)&\
                                                                 (final_dataset_3['Timeframe']==t_frame)&\
                                                                 (final_dataset_3['Probability__Good'] >= cut_off)]['Profit']).sum()

                p_table.loc[i,'Base Profit'] = (final_dataset_3[(final_dataset_3['Condition']==cond)&\
                                                                (final_dataset_3['Currency']==curr)&\
                                                                (final_dataset_3['Timeframe']==t_frame)]['Profit']).sum()

                p_table.loc[i,'Percent Profit Covered'] = p_table.loc[i,'Model Profit']/p_table.loc[i,'Base Profit']

                p_table.loc[i,'Predicted 1 Accuracy'] = p_table.loc[i,'TP']/(p_table.loc[i,'TP'] + p_table.loc[i,'FP'])

                p_table.loc[i,'Base_1s'] = len(final_dataset_3[(final_dataset_3['Condition']==cond)&\
                                                               (final_dataset_3['Currency']==curr)&\
                                                               (final_dataset_3['Timeframe']==t_frame)&\
                                                               (final_dataset_3['Actual_Good_Bad'] == 1)]\
                                               [final_dataset_3['Probability__Good'] >= cut_off])

                p_table.loc[i,'Base_0s'] = len(final_dataset_3[(final_dataset_3['Condition']==cond)&\
                                                               (final_dataset_3['Currency']==curr)&\
                                                               (final_dataset_3['Timeframe']==t_frame)&\
                                                               (final_dataset_3['Actual_Good_Bad'] == 0)]\
                                               [final_dataset_3['Probability__Good'] >= cut_off])

                p_table.loc[i,'Base Accuracy'] = (p_table.loc[i,'TP'] + p_table.loc[i,'FN'])/(p_table.loc[i,'TN'] + p_table.loc[i,'FP'] + p_table.loc[i,'FN'] + p_table.loc[i,'TP'])

                p_table.loc[i,'Base Trades Pass Through'] = p_table.loc[i,'Base_1s'] + p_table.loc[i,'Base_0s']
                
                p_table.loc[i,'Lift'] = p_table.loc[i,'Percent Profit Covered'] - p_table.loc[i,'Trades Pass Through']

                i+=1
        
    return p_table

# ------Main Algorithm -----#

In [ ]:
for item_method,row_method in method_list.iterrows():
    
    #-----Assigning the variable names-----------------------------------------------------------------#
    technique = row_method['Technique']
    method = row_method['Method']
    
    method_type = row_method['Method_Type']
    cluster_req = row_method['Cluster_Requirement']
    rel_imp_req = row_method['Rel_Imp_Requirement']
    method_type_value = int(row_method['Method_Type_Value'])
    
    grid_search_req = row_method['Grid_Serach']
    
    strategy = row_method['Strategy']
    
    str_folder_name = str(row_method['Strategy_Folder_Name'])
    
    
    
    classifier_req = row_method['Classifier_List']
    
    iteration_name = row_method['Iteration_Name']
    
    method_flag_folder = row_method['Method_Flag']
    
    input_random_seed = int(row_method['Random_Seed'])
    input_tensorflow_seed = int(row_method['Tensorflow_Seed'])
    

    
    #--------------------------------------------------------------------------------------------------#
    
    print("Iteration_Name:-")
    print(str(strategy)+"---"+str(technique)+"---"+str(method_flag_folder)+"---"+\
          str(method_type)+"---"+str(cluster_req)+"---"+str(rel_imp_req)\
          +"---"+str(method_type_value)+"---"+str(grid_search_req)+"---"+str(classifier_req))
    
    
    
        
    #---------------------------------------------------------------------------------------------------#
    #----Searching the required file from the Strategy Folder -----#
    os.chdir(data_path_name)
    
    for filename in os.listdir():
        
        if (method_type == 'Normal')or(method_type == 'Best30')or(method_type == 'Best40')or\
        (method_type == 'Best50')or(method_type == 'Normal_Passing'):
            search_file = '*_model_standarised_data*'
            
        elif (method_type == 'Cluster')and(method_type_value == 10):
            search_file = '*cluster_10*'
            
        elif (method_type == 'Cluster')and(method_type_value == 20):
            search_file = '*cluster_20*'
            
        elif (method_type == 'Cluster')and(method_type_value == 30):
            search_file = '*cluster_30*'
            
        elif (method_type == 'Cluster')and(method_type_value == 40):
            search_file = '*cluster_40*'
            
        elif (method_type == 'Cluster')and(method_type_value == 50):
            search_file = '*cluster_50*'
            
           
        if fnmatch.fnmatch(filename, search_file):
            df2 = pd.read_csv(filename)
            print(filename)
    #--------------------------------------------------------------------------------------------------------#
    
    df2.drop(columns = ['Closing_Flag','Profit_Loss'],axis=1,inplace=True)
    df2.rename(columns = {'Year':'YEAR',
                          'Exit_closing_flag':'Close_Flag',
                         'Profit_Loss_v1':'Profit'},inplace=True)
    df2 = df2[df2['Strategy'] == strategy]
    df2.reset_index(inplace=True)
    df2.drop('index',axis=1,inplace=True)
    
            
    df2['date']=pd.to_datetime(df2['date'],format = '%Y-%m-%d %H:%M:%S')
    df2['Good_Bad_Flag'] = np.where(df2['Good_Bad_Flag'] == 1,'Good','Bad')
    
    
    
    if (strategy == 'ClientEma')or(strategy == 'Client')or(strategy == 'Ema'):
        df2.rename(columns = {'Take_Profit_Risk':'TP_Ratio'},inplace=True)
            
        itr_list = df2[['Currency','Condition','TimeFrame','TP_Ratio']].drop_duplicates()
        itr_list.reset_index(inplace=True)
        itr_list.drop('index',axis=1,inplace=True)
        final_iter_list = itr_list
        
            
        
    elif (strategy == "Ichimoku"):
        df2.rename(columns = {'Take_Profit_Risk':'TP_Ratio'},inplace=True)
        
        df2['TP_Ratio'] = (abs(df2['Take_Profit'] - df2['TPP']))/(abs(df2['Stop_Loss'] - df2['TPP']))
        #df2.insert(9,'TP_Ratio',(abs(df2['Take_Profit'] - df2['TPP']))/(abs(df2['Stop_Loss'] - df2['TPP'])))
        
        itr_list = df2[['Currency','Condition','TimeFrame']].drop_duplicates()
        itr_list.reset_index(inplace=True)
        itr_list.drop('index',axis=1,inplace=True)
        final_iter_list = itr_list
        
        
        
    if (strategy == 'Ema'):
        final_iter_list = final_iter_list[final_iter_list['Currency'] != 'ZAR/JPY']
        final_iter_list.reset_index(inplace=True)
        final_iter_list.drop("index",axis=1,inplace=True)
        
        
        
                
    
    
    #---------Importing the classifier list if required -------------------------------------------------#
    if classifier_req == 'Yes':
        class_p = classifier_path_name
        cl_list = pd.read_excel(class_p+str(strategy)+"_"+str(method)+"_Classifier_list.xlsx")
        
    #------------------------------------------------------------------------------------------------------------#
    
    #-----Importing the best variable list----------#
    if rel_imp_req == 'Yes':
        path_best_var = rel_imp_path_name
        best_variables = pd.read_csv(path_best_var+"All Strategy_Relative_importance.csv")
    
    #------------------------------------------------------------------------------------------------------------#
    #----Creation of Folder
    export_path_1 = folder_path_name+str(str_folder_name)+"/"+str(strategy)+"/"+str(technique)+"/"+str(method_flag_folder)+"/"+str(iteration_name)+"/Sav_Res" 
    os.makedirs(export_path_1)
    
    
    os.chdir(export_path_1+'/')
    
    
    #----------------------------------------------------------------------------------------------#
    
    
    p_df_final = pd.DataFrame()
    final_dataset_4 = pd.DataFrame()

    p_df_final_train = pd.DataFrame()
    final_dataset_4_train = pd.DataFrame()
    
    

    #-------------Running the iterations-------#
    for ix in range(0,1):
        print(ix)

        final_dataset_1 = pd.DataFrame()
        final_dataset_2 = pd.DataFrame()
        final_dataset_3 = pd.DataFrame()

        final_dataset_1_train = pd.DataFrame()
        final_dataset_2_train = pd.DataFrame()
        final_dataset_3_train = pd.DataFrame()
        
        if (technique == 'RandomForest'):
            rf_classifiers_data = pd.DataFrame(columns = ['Condition','Currency','TimeFrame','max_depth','max_features',\
                                                      'min_samples_leaf','n_estimators'])
        elif (technique == 'SVM'):
            rf_classifiers_data = pd.DataFrame(columns = ['Condition','Currency','TimeFrame','C','gamma','kernel'])
            
            




        for item,row in final_iter_list.iterrows():

            b_g_set = pd.DataFrame(columns = ['Condition','Currency','Timeframe','Iteration'])
            b_g_set_train = pd.DataFrame(columns = ['Condition','Currency','Timeframe','Iteration'])
            
            
            print("--------------------")
            print("Running of Strategy combinations:-")
            print(str(row['Condition'])+"---"+str(row['Currency'])+"---"+str(row['TimeFrame'])+"---"+str(item)+"---"+str(item))

            condition = row['Condition']
            currency = row['Currency']
            currency_name = currency.replace("/", "_")
            tf = row['TimeFrame']
            
            


            #-------------------------------------------------------------------#
            df = df2[(df2['Condition'] == condition)&(df2['Currency'] == currency)&(df2['TimeFrame'] == tf)]




            ##############################################################################################
            ##############################################################################################
            #----------------------------OS or WithoutOS ----------------------------------------#
            if (method == 'WithoutOS'):
                print("Method - "+str(method))

                if (method_type != 'Cluster'):
                    print("Method - "+str(method_type))
                    X_train = df[df['YEAR']<=2018].loc[:,'_volume':]
                    X_test = df[df['YEAR']>2018].loc[:,'_volume':]
                elif (method_type == 'Cluster'):
                    print("Method - "+str(method_type))
                    X_train = df[df['YEAR']<=2018].loc[:,'0':]
                    X_test = df[df['YEAR']>2018].loc[:,'0':]


                y_train = np.where(pd.DataFrame(df[(df['YEAR'] <= 2018)]['Good_Bad_Flag'])['Good_Bad_Flag']=='Bad',0,1)
                y_test  = np.where(pd.DataFrame(df[(df['YEAR'] > 2018)]['Good_Bad_Flag'])['Good_Bad_Flag']=='Bad',0,1)

                z_profit_train  = pd.DataFrame(df[(df['YEAR'] <= 2018)]['Profit'])
                z_profit_test  = pd.DataFrame(df[(df['YEAR'] > 2018)]['Profit'])

                R_R_flag_train = pd.DataFrame(df[(df['YEAR'] <= 2018)]['TP_Ratio'])
                R_R_flag_test = pd.DataFrame(df[(df['YEAR'] > 2018)]['TP_Ratio'])

                Date_train = pd.DataFrame(df[(df['YEAR'] <= 2018)]['date'])
                Date_test = pd.DataFrame(df[(df['YEAR'] > 2018)]['date'])


            elif (method == 'OS'):
                print("Method - "+str(method))
                #---------------------Oversampling ---------------------#
                data_interim = df[(df['date'] <= dt.datetime(2018,12,31))]


                Overall_Data = data_interim
                ones_data = Overall_Data[Overall_Data['Close_Flag'] != 'Stop_Loss_Reach']
                zeros_data = Overall_Data[Overall_Data['Close_Flag'] == 'Stop_Loss_Reach']

                if len(zeros_data) >= len(ones_data):
                    zeros_sample = zeros_data.sample(len(ones_data))
                    X = pd.concat([ones_data,zeros_sample],axis = 0,ignore_index=True)
                else:
                    X = df[(df['date'] <= dt.datetime(2018,12,31))]

                #-------------------------------------------------------------------#

                if (method_type != 'Cluster'):
                    print("Method - "+str(method_type))
                    X_train = X.loc[:,'_volume':]
                    X_test = df[df['YEAR']>2018].loc[:,'_volume':]
                elif (method_type == 'Cluster'):
                    print("Method - "+str(method_type))
                    X_train = X.loc[:,'0':]
                    X_test = df[df['YEAR']>2018].loc[:,'0':]


                y_train = np.where(pd.DataFrame(X['Good_Bad_Flag'])['Good_Bad_Flag']=='Bad',0,1)
                y_test  = np.where(pd.DataFrame(df[(df['YEAR'] > 2018)]['Good_Bad_Flag'])['Good_Bad_Flag']=='Bad',0,1)

                z_profit_train  = pd.DataFrame(X['Profit'])
                z_profit_test  = pd.DataFrame(df[(df['YEAR'] > 2018)]['Profit'])

                R_R_flag_train = pd.DataFrame(X['TP_Ratio'])
                R_R_flag_test = pd.DataFrame(df[(df['YEAR'] > 2018)]['TP_Ratio'])

                Date_train = pd.DataFrame(X['date'])
                Date_test = pd.DataFrame(df[(df['YEAR'] > 2018)]['date'])


            ##############################################################################################
            ##############################################################################################



            ##############################################################################################
            ##############################################################################################

            if (method_type == 'Normal'):
                print("Running for Normal")

                if (technique == 'RandomForest')and(grid_search_req == 'Full_Check')and(classifier_req == 'No'):
                    best_params = RandomForest_with_pickle_dump.Grid_Search(X_train, y_train,input_random_seed,
                                                                            currency_name,condition,tf)
                
                    
                    rf_classifiers_data.loc[item,'Condition'] = condition
                    rf_classifiers_data.loc[item,'Currency'] = currency
                    rf_classifiers_data.loc[item,'TimeFrame'] = tf
                    rf_classifiers_data.loc[item,'max_depth'] = best_params['max_depth']
                    rf_classifiers_data.loc[item,'max_features'] = best_params['max_features']
                    rf_classifiers_data.loc[item,'min_samples_leaf'] = best_params['min_samples_leaf']
                    rf_classifiers_data.loc[item,'n_estimators'] = best_params['n_estimators']

                    n_estimators=int(best_params['n_estimators'])
                    max_depth=int(best_params['max_depth'])
                    max_features=str(best_params['max_features'])
                    min_samples_leaf=int(best_params['min_samples_leaf'])
                    
                    

                    model_run = RandomForest_with_pickle_dump.RF_Model(n_estimators,max_depth,max_features,
                                                                       min_samples_leaf,X_train,y_train,X_test,
                                                                       input_random_seed,currency_name,
                                                                       condition,tf,strategy)
                      





            elif (method_type != 'Normal'):

                if (technique == 'RandomForest'):
                    df_classfiers = cl_list[(cl_list['Condition'] == condition)&(cl_list['Currency'] == currency)&\
                                    (cl_list['TimeFrame'] == tf)]


                if (rel_imp_req == 'Yes'):
                    rank = method_type_value
                    df_var = best_variables[(best_variables['Condition'] == condition)&\
                                            (best_variables['Currency'] == currency)&\
                                            (best_variables['TimeFrame'] == tf)&\
                                            (best_variables['Strategy'] == strategy)]
                    df_var_1 = df_var[df_var['Rank'] <= rank]
                    final_variables = df_var_1['Variable'].tolist()


                    X_train = X_train[final_variables]
                    X_test = X_test[final_variables]



                if (technique == 'RandomForest')and(grid_search_req == 'Parameters_Passing')and(classifier_req == 'Yes'):
                    n_estimators = int(df_classfiers['n_estimators'])
                    max_depth = int(df_classfiers['max_depth'])
                    max_features = df_classfiers['max_features'].tolist()[0]
                    min_samples_leaf = int(df_classfiers['min_samples_leaf'])

                    model_run = RandomForest_with_pickle_dump.RF_Model(n_estimators,max_depth,max_features,min_samples_leaf,X_train,y_train,X_test,input_random_seed,currency_name,condition,tf,strategy)

                elif (technique == 'NN')and(grid_search_req == 'No')and(classifier_req == 'No'):
                    model_run = NeuralNetwork_with_pickle_dump.NN_Model(X_train,y_train,X_test,input_random_seed,input_tensorflow_seed,currency_name,condition,tf,strategy)

                elif (technique == 'SVM')and(grid_search_req == 'Full_Check')and(classifier_req == 'No'):
                    model_run = SVM_with_pickle_dump.Grid_Search(X_train, y_train,X_test,input_random_seed,currency_name,condition,tf,strategy)
                    

            ##############################################################################################
            ##############################################################################################






            rfc_predict = model_run[0]
            rfc_predict_train = model_run[1]
            
            if (technique != 'NN'):
                permt_table = model_run[2]
                permt_table.to_excel(export_path_1+"/"+str(currency_name)+"_"+str(condition)+"_"+str(tf)+"_Permt_table.xlsx",index=False)
            

            #-----For Overalll 2019 & 2020-----#
            b_g_set['Actual_Good_Bad']  = y_test
            
            if (technique != 'NN'):
                b_g_set['Probability'+'__'+str('Bad')]  = rfc_predict[0]
                b_g_set['Probability'+'__'+str('Good')]  = rfc_predict[1]
            elif (technique == 'NN'):
                b_g_set['Probability'+'__'+str('Good')]  = rfc_predict[0]
            
            b_g_set['Profit'] = np.array(z_profit_test)[:,0]
            b_g_set['Risk_Reward_Ratio'] = np.array(R_R_flag_test)[:,0]
            b_g_set['date'] = np.array(Date_test)[:,0]
            b_g_set['Year'] = b_g_set['date'].dt.year
            b_g_set['Condition'] = condition
            b_g_set['Currency'] = currency
            b_g_set['Timeframe'] = tf
            b_g_set['Iteration']  = ix


            #----Training-----#
            b_g_set_train['Actual_Good_Bad']  = y_train

            if (technique != 'NN'):
                b_g_set_train['Probability'+'__'+str('Bad')]  = rfc_predict_train[0]
                b_g_set_train['Probability'+'__'+str('Good')]  = rfc_predict_train[1]
            elif (technique == 'NN'):
                b_g_set_train['Probability'+'__'+str('Good')]  = rfc_predict_train[0]

            b_g_set_train['Profit'] = np.array(z_profit_train)[:,0]
            b_g_set_train['Risk_Reward_Ratio'] = np.array(R_R_flag_train)[:,0]
            b_g_set_train['date'] = np.array(Date_train)[:,0]
            b_g_set_train['Year'] = b_g_set_train['date'].dt.year
            b_g_set_train['Condition'] = condition
            b_g_set_train['Currency'] = currency
            b_g_set_train['Timeframe'] = tf
            b_g_set_train['Iteration']  = ix



            #---final_dataset_3 -- to retreive probability of good and bad of all trades across curries---------#
            final_dataset_3 = pd.concat([final_dataset_3,b_g_set],axis = 0,ignore_index=True)
            final_dataset_3_train = pd.concat([final_dataset_3_train,b_g_set_train],axis = 0,ignore_index=True)



            #final_dataset_3.reset_index(inplace=True)
            final_dataset_3['index'] = final_dataset_3.index
            final_dataset_3_train['index'] = final_dataset_3_train.index



        final_dataset_4 = pd.concat([final_dataset_4,final_dataset_3],axis = 0,ignore_index=True)
        final_dataset_4_train = pd.concat([final_dataset_4_train,final_dataset_3_train],axis = 0,ignore_index=True)



        clear_output()
        ##################################################
        
        

    export_path = folder_path_name+str(str_folder_name)+"/"+str(strategy)+"/"+str(technique)+"/"+str(method_flag_folder)+"/"+str(iteration_name)+"/Itr_Res" 
    os.makedirs(export_path)


    if (technique=='RandomForest')and(grid_search_req == 'Full_Check')and(classifier_req == 'No'):
        classifier_path = classifier_path_name
        rf_classifiers_data.to_excel(classifier_path+str(strategy)+"_"+str(method)+"_"+"Classifier_list.xlsx",index=False)
        


    
    #--File -1---#
    print("File1")
    final_dataset_4['Decile'] = final_dataset_4.groupby(['Currency','Condition','Timeframe'])\
    ['Probability__Good'].transform(lambda x: pd.qcut(x.rank(method='first'), 5, labels=False))
    final_dataset_4.to_excel(export_path+"/"+"prob good bad TEST.xlsx",index=False)

    #--File -2---#
    print("File2")
    final_dataset_4_train.to_excel(export_path+"/"+"prob good bad TRAIN.xlsx",index=False)

    #--File -3 (Overall Discount Table)---#
    print("File3")
    file3 = pd.DataFrame()
    file3 = model_probab_vs_RR_Ratio(final_dataset_4)
    file3.to_excel(export_path+"/"+"Overall Disc table.xlsx",index=False)

    #--File -4 (Overall Percentile Table)---#
    print("File4")    
    file4 = pd.DataFrame()
    file4 = percentile_table(final_dataset_4)
    file4.to_excel(export_path+"/"+"Overall % Table.xlsx",index=False)


    #----------------------------------------------------------------------------------#
    #--File -5,6,7,8,9---#
    x1 = Ranks(final_dataset_4,final_dataset_4_train)

    x_test = x1[0]

    x_train = {}
    x_train[1] = pd.DataFrame()
    x_train[1] = x1[1]

    #--5--#
    print("File5")
    x_test.to_excel(export_path+"/"+"All Ranks TEST.xlsx",index=False)
    #--6--#
    print("File6")
    x_train[1].to_excel(export_path+"/"+"All Ranks TRAIN.xlsx",index=False)


    train_rank_1 = x_train[1][x_train[1]['Rank_wrt_Profit'] == 1]
    train_rank_1.reset_index(inplace = True)
    train_rank_1.drop('index',axis=1,inplace=True)

    year = ['Overall',2019,2020]
    merge_data = {}

    for i_year in year:
        if i_year == 'Overall':
            x_test_1 = x_test[x_test['Year'] == i_year]
        elif i_year == 2019:
            x_test_1 = x_test[x_test['Year'] == i_year]
        elif i_year == 2020:
            x_test_1 = x_test[x_test['Year'] == i_year]


        merge_data[i_year] = pd.merge(left=x_test_1,right=train_rank_1,left_on=['Currency','Condition','Timeframe','Threshold_Value'],right_on=['Currency','Condition','Timeframe','Threshold_Value'],how = 'inner')

    #--7--#
    print("File7")
    merge_data['Overall'].to_excel(export_path+"/"+"Rk1_Train Test_all.xlsx",index=False)
    #--8--#
    print("File8")
    merge_data[2019].to_excel(export_path+"/"+"Rk1_Train Test_19.xlsx",index=False)
    #--9--#
    print("File9")
    merge_data[2020].to_excel(export_path+"/"+"Rk1_Train Test_20.xlsx",index=False)

    #-----File 10 (Discount Table by Currency & Condition)-------------------------------------------#
    print("File10")
    file10 = pd.DataFrame()
    file10 = Curr_Cond_Model_Probab_vs_RR_Ratio(final_dataset_4)
    file10.to_excel(export_path+"/"+"CurrCond Discount table.xlsx",index=False)

    #-----File 11 (Percentile Table by Currency & Condition)--------------------------------------#
    print("File11")
    file11 = pd.DataFrame()
    file11 = curr_cond_percentile_table(final_dataset_4)
    file11 = file11.sort_values(by = ['Year','Percentile','Currency','Condition','Timeframe'])

    file11.to_excel(export_path+"/"+"CurrCond %tile Table.xlsx",index=False)
    
    
    
    
    ###########################################################################
    ######################  Exporting the Lift Chart    #######################
    
    #------calculation of decile -----#
    print("Lift_Chart")
    
    final_dataset_4['Decile'] = final_dataset_4.groupby(['Currency','Condition','Timeframe'])\
    ['Probability__Good'].transform(lambda x: pd.qcut(x.rank(method='first'), 5, labels=False))
    




    #-----Creation of Pivot Table for binning Ones & Zeroes-----#
    table = pd.pivot_table(final_dataset_4,index=["Currency","Condition","Timeframe","Decile"],\
                           columns=["Actual_Good_Bad"],aggfunc={"Actual_Good_Bad":len})
    table.reset_index(inplace = True)
    table = table.fillna(0)


    #---Renaming the column name of pivot table -----#
    table.columns = ['_'.join(str(s).strip() for s in col if s) for col in table.columns]


    #----Renaming the Column to "Count of Ones"-----#
    table = table[['Currency','Condition','Timeframe','Decile','Actual_Good_Bad_1']]
    table.rename(columns={'Actual_Good_Bad_1':'Count of Ones'},inplace=True)



    #-----Calculating the average probability of each decile ----#
    average_probab = final_dataset_4.groupby(by = ['Currency','Condition','Timeframe','Decile'])['Probability__Good'].mean()
    average_probab = pd.DataFrame(average_probab)
    average_probab.reset_index(inplace=True)
    average_probab.rename(columns = {'Probability__Good':'Avg of Probab_Good in the Decile'},inplace=True)


    #-----Merging the average probab value with the df_Table -----#
    table = pd.merge(left=table,right=average_probab,left_on=['Currency','Condition','Timeframe','Decile'],\
                     right_on=['Currency','Condition','Timeframe','Decile'],how = 'left')




    #----Sorting wrt to DecileRank ------#
    table = table.groupby(['Currency','Condition','Timeframe']).apply(lambda x: x.sort_values(['Decile'], ascending=False).drop(['Currency','Condition','Timeframe'], axis=1))
    table.reset_index(inplace=True)
    table.drop('level_3',axis=1,inplace=True)



    #------Calculation of total ones----#
    total_counts = table.groupby(['Currency','Condition','Timeframe'])['Count of Ones'].agg('sum')
    total_counts = pd.DataFrame(total_counts)
    total_counts.rename(columns={'Count of Ones':'Total Ones'},inplace=True)


    table = pd.merge(left=table,right=total_counts,left_on=['Currency','Condition','Timeframe'],\
                     right_on=['Currency','Condition','Timeframe'],how = 'left')




    #------Calcultion of Percent of Good Trades,Random Sample,Lift ------#

    table['Cumsum Ones'] = table.groupby(by=['Currency','Condition','Timeframe'])['Count of Ones'].cumsum()
    table['% of Good Trades'] = table['Cumsum Ones']/table['Total Ones']
    table['sequence'] = table.groupby(['Currency','Condition','Timeframe']).cumcount()+1
    table['Random Sample'] = table['sequence']/5
    table['Lift'] = table['% of Good Trades'] - table['Random Sample']
    
    
    
    ########################---Profit Lift----##############################
    #----Total Trades -----#
    total_trades = final_dataset_4.groupby(['Currency','Condition','Timeframe','Decile'])['Profit'].count()
    total_trades = pd.DataFrame(total_trades)
    total_trades.reset_index(inplace=True)
    total_trades.rename(columns = {'Profit':'Total Trades per Decile'},inplace = True)
    
    table = pd.merge(left=table,right=total_trades,left_on=['Currency','Condition','Timeframe','Decile'],\
                     right_on=['Currency','Condition','Timeframe','Decile'],how = 'left')

    
    
    decile_profit = final_dataset_4.groupby(['Currency','Condition','Timeframe','Decile'])['Profit'].sum()
    decile_profit = pd.DataFrame(decile_profit)
    decile_profit.reset_index(inplace=True)
    decile_profit.rename(columns = {'Profit':'Decile Profit'},inplace = True)
    
    table = pd.merge(left=table,right=decile_profit,left_on=['Currency','Condition','Timeframe','Decile'],\
                     right_on=['Currency','Condition','Timeframe','Decile'],how = 'left')
    
    
    table['Cummulative Profit'] = table.groupby(by=['Currency','Condition','Timeframe'])['Decile Profit'].cumsum()
    total_profit = table.groupby(by=['Currency','Condition','Timeframe'])['Decile Profit'].sum()
    total_profit = pd.DataFrame(total_profit)
    total_profit.rename(columns = {'Decile Profit':'Total Profit'},inplace=True)

    table = pd.merge(left=table,right=total_profit,left_on=['Currency','Condition','Timeframe'],\
                         right_on=['Currency','Condition','Timeframe'],how = 'left')

    table['% Profit Covered'] = table['Cummulative Profit']/table['Total Profit']
    table['Profit Lift'] = table['% Profit Covered'] - table['Random Sample']
    
    ########################################################################




    #-------Filtering the required columns -----#
    table = table[['Currency','Condition','Timeframe','Decile','Avg of Probab_Good in the Decile',
                   'Count of Ones','Total Ones','Total Trades per Decile','% of Good Trades','Random Sample','Lift',
                  'Decile Profit','Cummulative Profit','Total Profit','% Profit Covered','Profit Lift']]

    
    table.to_excel(export_path+"/"+"Lift Chart.xlsx",index=False)
    
    ###########################################################################
    ###########################################################################
    
